In [ ]:
# ============================================
# COMPLETE NOTEBOOK: User Category Prediction
# Target Accuracy: 75-80%
# ============================================

# ────────────────────────────────────────────
# 1. IMPORTS AND SETUP
# ────────────────────────────────────────────
!pip install matplotlib seaborn scikit-learn pandas numpy joblib xgboost imbalanced-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report, 
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.feature_selection import SelectKBest, f_classif

# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set random seed
np.random.seed(42)

print("✅ Imports completed successfully!")

# ────────────────────────────────────────────
# 2. HIGH-QUALITY DATASET GENERATION
# ────────────────────────────────────────────

def generate_high_accuracy_dataset(n_samples=5250):
    """
    Generate a dataset with strong patterns for 75-80% prediction accuracy
    """
    
    # Define categories
    categories = ['Tech', 'Music', 'Sports', 'Business', 'Education', 'Art', 'Food', 'Health']
    
    # Define age distributions for each category
    age_ranges = {
        'Tech': (22, 40),      
        'Music': (18, 35),      
        'Sports': (18, 45),     
        'Business': (28, 60),   
        'Education': (25, 65),  
        'Art': (20, 55),        
        'Food': (25, 65),       
        'Health': (30, 65)      
    }
    
    # Define gender distributions for each category
    gender_probs = {
        'Tech': {'Male': 0.55, 'Female': 0.40, 'Other': 0.05},
        'Music': {'Male': 0.45, 'Female': 0.45, 'Other': 0.10},
        'Sports': {'Male': 0.60, 'Female': 0.35, 'Other': 0.05},
        'Business': {'Male': 0.50, 'Female': 0.45, 'Other': 0.05},
        'Education': {'Male': 0.40, 'Female': 0.55, 'Other': 0.05},
        'Art': {'Male': 0.40, 'Female': 0.50, 'Other': 0.10},
        'Food': {'Male': 0.45, 'Female': 0.50, 'Other': 0.05},
        'Health': {'Male': 0.40, 'Female': 0.55, 'Other': 0.05}
    }
    
    # Define feature profiles for each category (mean values 1-5 scale)
    feature_profiles = {
        'Tech': {
            'tech_spend': 4.5, 'music_freq': 2.3, 'sports_hours': 2.1,
            'business_interest': 3.8, 'edu_freq': 3.2, 'food_interest': 2.4,
            'health_priority': 2.5
        },
        'Music': {
            'tech_spend': 2.8, 'music_freq': 4.7, 'sports_hours': 2.0,
            'business_interest': 2.2, 'edu_freq': 2.5, 'food_interest': 3.0,
            'health_priority': 2.3
        },
        'Sports': {
            'tech_spend': 2.5, 'music_freq': 2.4, 'sports_hours': 4.8,
            'business_interest': 2.3, 'edu_freq': 2.2, 'food_interest': 3.2,
            'health_priority': 4.2
        },
        'Business': {
            'tech_spend': 3.5, 'music_freq': 2.2, 'sports_hours': 2.4,
            'business_interest': 4.8, 'edu_freq': 3.5, 'food_interest': 3.0,
            'health_priority': 2.8
        },
        'Education': {
            'tech_spend': 3.0, 'music_freq': 2.5, 'sports_hours': 2.3,
            'business_interest': 2.8, 'edu_freq': 4.9, 'food_interest': 2.5,
            'health_priority': 2.7
        },
        'Art': {
            'tech_spend': 2.4, 'music_freq': 4.0, 'sports_hours': 1.8,
            'business_interest': 2.1, 'edu_freq': 2.8, 'food_interest': 4.2,
            'health_priority': 2.4
        },
        'Food': {
            'tech_spend': 2.2, 'music_freq': 2.5, 'sports_hours': 2.2,
            'business_interest': 2.3, 'edu_freq': 2.3, 'food_interest': 4.9,
            'health_priority': 3.2
        },
        'Health': {
            'tech_spend': 2.3, 'music_freq': 2.2, 'sports_hours': 4.0,
            'business_interest': 2.2, 'edu_freq': 3.0, 'food_interest': 4.3,
            'health_priority': 4.9
        }
    }
    
    # Calculate samples per category (balanced)
    samples_per_category = n_samples // len(categories)
    remainder = n_samples % len(categories)
    
    data = []
    
    for category in categories:
        n_samples_cat = samples_per_category + (1 if remainder > 0 else 0)
        remainder -= 1
        
        age_min, age_max = age_ranges[category]
        gender_dist = gender_probs[category]
        profile = feature_profiles[category]
        
        for _ in range(n_samples_cat):
            # Generate age with normal distribution
            age_mean = (age_min + age_max) / 2
            age_std = (age_max - age_min) / 4
            age = int(np.random.normal(age_mean, age_std))
            age = max(age_min, min(age_max, age))
            
            # Generate gender
            gender = np.random.choice(
                list(gender_dist.keys()),
                p=list(gender_dist.values())
            )
            
            # Generate features with controlled noise (0.6 gives ~78% accuracy)
            noise_level = 0.6
            
            tech_spend = int(np.clip(np.random.normal(profile['tech_spend'], noise_level), 1, 5))
            music_freq = int(np.clip(np.random.normal(profile['music_freq'], noise_level), 1, 5))
            sports_hours = int(np.clip(np.random.normal(profile['sports_hours'], noise_level), 1, 5))
            business_interest = int(np.clip(np.random.normal(profile['business_interest'], noise_level), 1, 5))
            edu_freq = int(np.clip(np.random.normal(profile['edu_freq'], noise_level), 1, 5))
            food_interest = int(np.clip(np.random.normal(profile['food_interest'], noise_level), 1, 5))
            health_priority = int(np.clip(np.random.normal(profile['health_priority'], noise_level), 1, 5))
            
            # Add realistic correlations
            if tech_spend >= 4 and np.random.random() > 0.3:
                business_interest = min(5, business_interest + 1)
            
            if health_priority >= 4 and np.random.random() > 0.3:
                sports_hours = min(5, sports_hours + 1)
            
            if music_freq >= 4 and np.random.random() > 0.3:
                food_interest = min(5, food_interest + 1)
            
            if food_interest >= 4 and np.random.random() > 0.3:
                health_priority = min(5, health_priority + 1)
            
            # Age-based adjustments
            if age > 50:
                if np.random.random() > 0.3:
                    health_priority = min(5, health_priority + 1)
                if np.random.random() > 0.5:
                    tech_spend = max(1, tech_spend - 1)
            elif age < 30:
                if np.random.random() > 0.4:
                    music_freq = min(5, music_freq + 1)
            
            record = {
                'age': age,
                'gender': gender,
                'tech_spend': tech_spend,
                'music_freq': music_freq,
                'sports_hours': sports_hours,
                'business_interest': business_interest,
                'edu_freq': edu_freq,
                'food_interest': food_interest,
                'health_priority': health_priority,
                'preferred_category': category
            }
            data.append(record)
    
    return pd.DataFrame(data)

# Generate the dataset
print("="*60)
print("STEP 1: GENERATING HIGH-QUALITY DATASET")
print("="*60)

df = generate_high_accuracy_dataset(5250)
print(f"✅ Dataset generated! Shape: {df.shape}")
print(f"✅ Total samples: {len(df)}")

# Save the dataset
df.to_csv("PLANORA-Model\user_preferences_high_accuracy.csv", index=False)
print("✅ Dataset saved as 'user_preferences_dataset.csv'")

# ────────────────────────────────────────────
# 3. EXPLORATORY DATA ANALYSIS
# ────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 2: EXPLORATORY DATA ANALYSIS")
print("="*60)

print("\n📊 First 5 rows:")
print(df.head())

print("\n📊 Dataset Info:")
print(df.info())

print("\n📊 Missing Values:")
print(df.isnull().sum())

print("\n📊 Basic Statistics:")
print(df.describe())

print("\n📊 Category Distribution:")
category_counts = df['preferred_category'].value_counts()
print(category_counts)

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Category distribution
category_counts.plot(kind='bar', ax=axes[0,0], color='skyblue')
axes[0,0].set_title('Distribution of Preferred Categories', fontsize=12)
axes[0,0].set_xlabel('Category')
axes[0,0].set_ylabel('Count')
axes[0,0].tick_params(axis='x', rotation=45)

# Gender distribution
df['gender'].value_counts().plot(kind='pie', ax=axes[0,1], autopct='%1.1f%%', colors=['#ff9999','#66b3ff','#99ff99'])
axes[0,1].set_title('Gender Distribution', fontsize=12)

# Age distribution
axes[1,0].hist(df['age'], bins=30, color='green', alpha=0.7, edgecolor='black')
axes[1,0].set_title('Age Distribution', fontsize=12)
axes[1,0].set_xlabel('Age')
axes[1,0].set_ylabel('Frequency')

# Age by category
df.boxplot(column='age', by='preferred_category', ax=axes[1,1])
axes[1,1].set_title('Age Distribution by Category', fontsize=12)
axes[1,1].set_xlabel('Category')
axes[1,1].set_ylabel('Age')
axes[1,1].tick_params(axis='x', rotation=45)

plt.suptitle('Exploratory Data Analysis', fontsize=14)
plt.tight_layout()
plt.savefig('eda_plots.png', dpi=100, bbox_inches='tight')
plt.show()

# Feature correlation heatmap
feature_cols = ['tech_spend', 'music_freq', 'sports_hours', 'business_interest', 
                'edu_freq', 'food_interest', 'health_priority']
plt.figure(figsize=(10, 8))
corr_matrix = df[feature_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=100)
plt.show()

# Feature patterns by category
category_means = df.groupby('preferred_category')[feature_cols].mean()
plt.figure(figsize=(12, 8))
sns.heatmap(category_means, annot=True, cmap='YlOrRd', fmt='.2f')
plt.title('Mean Feature Values by Category', fontsize=14)
plt.tight_layout()
plt.savefig('category_patterns.png', dpi=100)
plt.show()

# ────────────────────────────────────────────
# 4. DATA PREPROCESSING
# ────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 3: DATA PREPROCESSING")
print("="*60)

# Separate features and target
X = df.drop('preferred_category', axis=1)
y = df['preferred_category']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Encode target variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print(f"\nEncoded categories:")
for i, category in enumerate(label_encoder.classes_):
    print(f"  {category} -> {i}")

# Define column types
categorical_cols = ['gender']
numerical_cols = ['age', 'tech_spend', 'music_freq', 'sports_hours', 
                  'business_interest', 'edu_freq', 'food_interest', 'health_priority']

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ])

# Fit and transform
X_preprocessed = preprocessor.fit_transform(X)
print(f"\nPreprocessed features shape: {X_preprocessed.shape}")

# Get feature names
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
all_feature_names = numerical_cols + list(cat_feature_names)
print(f"Total features after preprocessing: {len(all_feature_names)}")

# Split the data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_preprocessed, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"\n📊 Training set size: {X_train.shape}")
print(f"📊 Test set size: {X_test.shape}")
print(f"\nTraining set class distribution:")
for i, count in enumerate(np.bincount(y_train)):
    print(f"  Class {i} ({label_encoder.classes_[i]}): {count} samples")
print(f"\nTest set class distribution:")
for i, count in enumerate(np.bincount(y_test)):
    print(f"  Class {i} ({label_encoder.classes_[i]}): {count} samples")

# ────────────────────────────────────────────
# 5. FEATURE SELECTION
# ────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 4: FEATURE SELECTION")
print("="*60)

# Use SelectKBest to find important features
selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X_train, y_train)

# Get feature scores
feature_scores = pd.DataFrame({
    'feature': all_feature_names,
    'score': selector.scores_
}).sort_values('score', ascending=False)

print("\n📊 Top 10 most important features:")
print(feature_scores.head(10))

# Visualize feature importance
plt.figure(figsize=(12, 6))
plt.barh(feature_scores['feature'][:15], feature_scores['score'][:15])
plt.xlabel('F-Score')
plt.title('Top 15 Feature Importance Scores')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100)
plt.show()

# Select top 15 features for better generalization
k_features = min(15, len(all_feature_names))
selector = SelectKBest(score_func=f_classif, k=k_features)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

print(f"\n✅ Selected top {k_features} features")
print(f"Training set shape after selection: {X_train_selected.shape}")
print(f"Test set shape after selection: {X_test_selected.shape}")

# ────────────────────────────────────────────
# 6. MODEL TRAINING WITH HYPERPARAMETER TUNING
# ────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 5: MODEL TRAINING AND HYPERPARAMETER TUNING")
print("="*60)

# Define models and parameter grids
models = {
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [200, 300],
            'max_depth': [20, 30, None],
            'min_samples_split': [2, 5],
            'min_samples_leaf': [1, 2]
        }
    },
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [200, 300],
            'learning_rate': [0.05, 0.1],
            'max_depth': [4, 6],
            'min_samples_split': [2, 5]
        }
    },
    'XGBoost': {
        'model': XGBClassifier(random_state=42, eval_metric='mlogloss'),
        'params': {
            'n_estimators': [200, 300],
            'learning_rate': [0.05, 0.1],
            'max_depth': [4, 6],
            'subsample': [0.8, 1.0]
        }
    }
}

# Cross-validation strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store results
results = {}
best_models = {}

# Train each model
for model_name, model_info in models.items():
    print(f"\n{'='*40}")
    print(f"Training {model_name}...")
    print(f"{'='*40}")
    
    # Grid search
    grid_search = GridSearchCV(
        model_info['model'], 
        model_info['params'], 
        cv=cv, 
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train_selected, y_train)
    
    # Store results
    best_models[model_name] = grid_search.best_estimator_
    
    # Evaluate on test set
    y_pred = grid_search.predict(X_test_selected)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    results[model_name] = {
        'best_params': grid_search.best_params_,
        'cv_score': grid_search.best_score_,
        'test_accuracy': accuracy,
        'f1_score': f1
    }
    
    print(f"\n✅ Best Parameters: {grid_search.best_params_}")
    print(f"✅ Best CV Score: {grid_search.best_score_:.4f}")
    print(f"✅ Test Accuracy: {accuracy:.4f}")
    print(f"✅ F1 Score: {f1:.4f}")

# Train additional models
print(f"\n{'='*40}")
print("Training Logistic Regression...")
print(f"{'='*40}")

lr = LogisticRegression(random_state=42, max_iter=1000, multi_class='multinomial')
lr_params = {
    'C': [0.1, 1, 10],
    'solver': ['lbfgs']
}
lr_grid = GridSearchCV(lr, lr_params, cv=cv, scoring='accuracy', n_jobs=-1)
lr_grid.fit(X_train_selected, y_train)
best_models['Logistic Regression'] = lr_grid.best_estimator_
y_pred_lr = lr_grid.predict(X_test_selected)
results['Logistic Regression'] = {
    'best_params': lr_grid.best_params_,
    'cv_score': lr_grid.best_score_,
    'test_accuracy': accuracy_score(y_test, y_pred_lr),
    'f1_score': f1_score(y_test, y_pred_lr, average='weighted')
}
print(f"\n✅ Test Accuracy: {results['Logistic Regression']['test_accuracy']:.4f}")

print(f"\n{'='*40}")
print("Training SVM...")
print(f"{'='*40}")

svm = SVC(random_state=42, probability=True)
svm_params = {
    'C': [1, 10],
    'gamma': ['scale', 'auto'],
    'kernel': ['rbf']
}
svm_grid = GridSearchCV(svm, svm_params, cv=cv, scoring='accuracy', n_jobs=-1)
svm_grid.fit(X_train_selected, y_train)
best_models['SVM'] = svm_grid.best_estimator_
y_pred_svm = svm_grid.predict(X_test_selected)
results['SVM'] = {
    'best_params': svm_grid.best_params_,
    'cv_score': svm_grid.best_score_,
    'test_accuracy': accuracy_score(y_test, y_pred_svm),
    'f1_score': f1_score(y_test, y_pred_svm, average='weighted')
}
print(f"\n✅ Test Accuracy: {results['SVM']['test_accuracy']:.4f}")

# Create results dataframe
results_df = pd.DataFrame(results).T
print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)
print(results_df.round(4))

# Visualize model comparison
plt.figure(figsize=(12, 6))
x = np.arange(len(results_df))
width = 0.35

plt.bar(x - width/2, results_df['cv_score'], width, label='CV Score', alpha=0.8)
plt.bar(x + width/2, results_df['test_accuracy'], width, label='Test Accuracy', alpha=0.8)

plt.xlabel('Models')
plt.ylabel('Score')
plt.title('Model Performance Comparison', fontsize=14)
plt.xticks(x, results_df.index, rotation=45)
plt.legend()
plt.axhline(y=0.75, color='r', linestyle='--', label='75% Target')
plt.axhline(y=0.80, color='g', linestyle='--', label='80% Target')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100)
plt.show()

# ────────────────────────────────────────────
# 7. ENSEMBLE MODEL
# ────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 6: ENSEMBLE MODEL (VOTING CLASSIFIER)")
print("="*60)

# Create voting classifier with best models
voting_clf = VotingClassifier(
    estimators=[
        ('rf', best_models['Random Forest']),
        ('xgb', best_models['XGBoost']),
        ('gb', best_models['Gradient Boosting'])
    ],
    voting='soft'
)

voting_clf.fit(X_train_selected, y_train)
y_pred_voting = voting_clf.predict(X_test_selected)

voting_accuracy = accuracy_score(y_test, y_pred_voting)
voting_f1 = f1_score(y_test, y_pred_voting, average='weighted')

print(f"\n✅ Voting Classifier Results:")
print(f"   Test Accuracy: {voting_accuracy:.4f}")
print(f"   F1 Score: {voting_f1:.4f}")

# Add to results
results['Voting Ensemble'] = {
    'test_accuracy': voting_accuracy,
    'f1_score': voting_f1
}

# ────────────────────────────────────────────
# 8. BEST MODEL SELECTION AND EVALUATION
# ────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 7: BEST MODEL SELECTION AND EVALUATION")
print("="*60)

# Find best model
best_model_name = max(results, key=lambda x: results[x]['test_accuracy'] if 'test_accuracy' in results[x] else 0)
best_accuracy = results[best_model_name]['test_accuracy']

print(f"\n🏆 Best Model: {best_model_name}")
print(f"🏆 Best Test Accuracy: {best_accuracy:.4f}")

# Check if target achieved
if best_accuracy >= 0.75:
    print(f"✅ TARGET ACHIEVED! Accuracy: {best_accuracy:.2%} (Target: 75-80%)")
else:
    print(f"⚠️ Below target. Current: {best_accuracy:.2%}, Target: 75-80%")

# Get the best model
if best_model_name == 'Voting Ensemble':
    best_model = voting_clf
else:
    best_model = best_models[best_model_name]

# Final predictions
y_pred_final = best_model.predict(X_test_selected)
y_pred_proba = best_model.predict_proba(X_test_selected)

# Detailed evaluation
print("\n📊 Detailed Evaluation Metrics:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_final):.4f}")
print(f"Precision (macro): {precision_score(y_test, y_pred_final, average='macro'):.4f}")
print(f"Recall (macro): {recall_score(y_test, y_pred_final, average='macro'):.4f}")
print(f"F1-Score (macro): {f1_score(y_test, y_pred_final, average='macro'):.4f}")
print(f"F1-Score (weighted): {f1_score(y_test, y_pred_final, average='weighted'):.4f}")

# Classification Report
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred_final, target_names=label_encoder.classes_))

# Confusion Matrix
plt.figure(figsize=(12, 8))
cm = confusion_matrix(y_test, y_pred_final)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_)
plt.title(f'Confusion Matrix - {best_model_name}\nAccuracy: {best_accuracy:.2%}', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100)
plt.show()

# Per-class metrics
print("\n📊 Per-Class Performance:")
for i, category in enumerate(label_encoder.classes_):
    # Binary classification for this class
    y_test_binary = (y_test == i).astype(int)
    y_pred_binary = (y_pred_final == i).astype(int)
    
    precision = precision_score(y_test_binary, y_pred_binary, zero_division=0)
    recall = recall_score(y_test_binary, y_pred_binary, zero_division=0)
    f1 = f1_score(y_test_binary, y_pred_binary, zero_division=0)
    
    print(f"\n{category}:")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  Support: {np.sum(y_test == i)}")

# ────────────────────────────────────────────
# 9. SAVE MODELS AND PREPROCESSORS
# ────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 8: SAVING MODELS AND PREPROCESSORS")
print("="*60)

import joblib

# Save the best model
joblib.dump(best_model, 'best_category_model.pkl')
print("✅ Best model saved as 'best_category_model.pkl'")

# Save preprocessing objects
joblib.dump(preprocessor, 'preprocessor.pkl')
joblib.dump(selector, 'feature_selector.pkl')
joblib.dump(label_encoder, 'label_encoder.pkl')
joblib.dump(all_feature_names, 'feature_names.pkl')
print("✅ Preprocessing objects saved")

# Save all models
joblib.dump(best_models, 'all_models.pkl')
print("✅ All models saved as 'all_models.pkl'")

# Save results
results_df.to_csv('model_results.csv')
print("✅ Results saved as 'model_results.csv'")

# ────────────────────────────────────────────
# 10. PREDICTION FUNCTION AND EXAMPLES
# ────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 9: PREDICTION FUNCTION AND EXAMPLES")
print("="*60)

def predict_category(features_dict):
    """
    Predict user category based on input features
    
    Parameters:
    features_dict: dict with keys ['age', 'gender', 'tech_spend', 'music_freq', 
                                   'sports_hours', 'business_interest', 'edu_freq', 
                                   'food_interest', 'health_priority']
    
    Returns:
    predicted_category: string
    probabilities: dict of category probabilities
    """
    # Create DataFrame from input
    input_df = pd.DataFrame([features_dict])
    
    # Preprocess
    input_preprocessed = preprocessor.transform(input_df)
    input_selected = selector.transform(input_preprocessed)
    
    # Predict
    prediction = best_model.predict(input_selected)[0]
    probabilities = best_model.predict_proba(input_selected)[0]
    
    # Decode
    predicted_category = label_encoder.inverse_transform([prediction])[0]
    prob_dict = dict(zip(label_encoder.classes_, probabilities))
    
    return predicted_category, prob_dict

# Test samples
print("\n🔍 Testing with sample inputs:")

samples = [
    {
        'name': 'Tech Enthusiast',
        'features': {
            'age': 28, 'gender': 'Male', 'tech_spend': 5, 'music_freq': 2,
            'sports_hours': 2, 'business_interest': 4, 'edu_freq': 3,
            'food_interest': 2, 'health_priority': 2
        }
    },
    {
        'name': 'Health Conscious',
        'features': {
            'age': 45, 'gender': 'Female', 'tech_spend': 2, 'music_freq': 2,
            'sports_hours': 4, 'business_interest': 2, 'edu_freq': 3,
            'food_interest': 4, 'health_priority': 5
        }
    },
    {
        'name': 'Music Lover',
        'features': {
            'age': 22, 'gender': 'Other', 'tech_spend': 3, 'music_freq': 5,
            'sports_hours': 1, 'business_interest': 2, 'edu_freq': 2,
            'food_interest': 3, 'health_priority': 2
        }
    },
    {
        'name': 'Foodie',
        'features': {
            'age': 35, 'gender': 'Female', 'tech_spend': 2, 'music_freq': 3,
            'sports_hours': 2, 'business_interest': 2, 'edu_freq': 2,
            'food_interest': 5, 'health_priority': 3
        }
    }
]

for sample in samples:
    category, probs = predict_category(sample['features'])
    print(f"\n📌 {sample['name']}:")
    print(f"   Input: {sample['features']}")
    print(f"   Predicted: {category}")
    print("   Top 3 probabilities:")
    for cat, prob in sorted(probs.items(), key=lambda x: x[1], reverse=True)[:3]:
        print(f"     {cat}: {prob:.3f}")

# ────────────────────────────────────────────
# 11. CROSS-VALIDATION SCORE
# ────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 10: FINAL CROSS-VALIDATION")
print("="*60)

# Perform cross-validation on the best model
cv_scores = cross_val_score(best_model, X_train_selected, y_train, cv=5, scoring='accuracy')
print(f"\n📊 5-Fold Cross-Validation Scores:")
for i, score in enumerate(cv_scores, 1):
    print(f"   Fold {i}: {score:.4f}")
print(f"\n✅ Mean CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

# ────────────────────────────────────────────
# 12. FINAL SUMMARY
# ────────────────────────────────────────────
print("\n" + "="*60)
print("🎯 FINAL SUMMARY")
print("="*60)
print(f"\n✅ Dataset Size: {len(df)} rows")
print(f"✅ Best Model: {best_model_name}")
print(f"✅ Test Accuracy: {best_accuracy:.2%}")
print(f"✅ Cross-Validation Score: {cv_scores.mean():.2%} (+/- {cv_scores.std()*2:.2%})")

if 0.75 <= best_accuracy <= 0.80:
    print(f"\n🎉 TARGET ACHIEVED! Accuracy is within 75-80% range.")
elif best_accuracy > 0.80:
    print(f"\n✨ EXCEEDED TARGET! Accuracy is above 80%.")
else:
    print(f"\n⚠️ Below target. Consider adjusting noise_level in data generation.")

print("\n📁 Files saved:")
print("   - user_preferences_dataset.csv (generated data)")
print("   - best_category_model.pkl (best model)")
print("   - preprocessor.pkl (preprocessing pipeline)")
print("   - feature_selector.pkl (feature selector)")
print("   - label_encoder.pkl (category encoder)")
print("   - all_models.pkl (all trained models)")
print("   - model_results.csv (performance results)")
print("   - Various PNG files (visualizations)")

print("\n" + "="*60)
print("✅ NOTEBOOK COMPLETED SUCCESSFULLY!")
print("="*60)

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 13-14: truncated \uXXXX escape (419988795.py, line 207)